## IMPORTACION
Librerias, funciones, conexion

In [ ]:
import mysql.connector
import pandas as pd
from datetime import datetime
import calendar
from dateutil.relativedelta import relativedelta
from pathlib import Path


In [ ]:
from utils import (
    obtener_inicio_mes_siguiente, #Inicio-fin del mes siguiente
    ejecutar_y_guardar,
    obtener_mes_anterior,  #Inicio-fin mes
    guardar_csv,
    obtener_quincena_anterior,
    obtener_trimestre_anterior 
)

In [ ]:
conn = mysql.connector.connect(
    host = "datamart.mex.internal.lyftbikes.com",
    port =3306,
    database= "mex_datawarehouse_bss4",
    user= "dm_mex_ge",
    password= "m+y#J9JI9F+^4qjSJLu^R",
)

conn.cursor().execute(
    "SET SESSION max_execution_time = 300000"
)

## FECHA INICIO Y FIN
Fechas que esta tomando en cuenta la consulta

In [ ]:
inicio_mes = obtener_inicio_mes_siguiente()
print(inicio_mes)

### Ruta
Agregar ruta y nombre del archivo

In [ ]:
ruta_archivo_vencimientos = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
    rf"\vencimientos_{inicio_mes[:7]}.csv"
)

print(ruta_archivo_vencimientos)

## Query reporte vencimientos

In [ ]:
query_vencimientos =f"""
    SELECT
        member_accountNumber AS cuenta,
        accountHolderFirstName AS nombre,
        accountHolderLastName AS apellido,
        accountHolderEmail AS correo,
        CONVERT_TZ(
            FROM_UNIXTIME(end/1000,'%Y-%m-%d %H:%i:%s'),
            'UTC',
            'America/Costa_Rica'
        ) AS vencimiento

    FROM BikeSubscriptionFact s
    LEFT JOIN BikeAccountFact a
        ON s.member_accountnumber = a.publicAccountNumber

    WHERE (subscriptionType_id = 4 OR subscriptionType_id = 5)

      AND end/1000 BETWEEN
          UNIX_TIMESTAMP(
              CONVERT_TZ('{inicio_mes}','America/Costa_Rica','UTC')
          )

      AND UNIX_TIMESTAMP(
              CONVERT_TZ(
                  DATE_ADD('{inicio_mes}', INTERVAL 1 MONTH),
                  'America/Costa_Rica',
                  'UTC'
              )
          )

    ORDER BY vencimiento ASC
    """

In [ ]:
df_vencimiento_membresias = ejecutar_y_guardar(
    query_vencimientos,
    conn,
    ruta_archivo_vencimientos
)